In [ ]:
import sys
import torch

print("Python:", sys.version)
print("PyTorch:", torch.__version__)
print("CUDA:", torch.version.cuda)
print("GPU available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

Python: 3.13.15 (main, Aug  6 2026, 11:06:22) [GCC 13.3.0]
PyTorch: 2.11.0+cu130
CUDA: 13.0
GPU available: True
GPU: Tesla T4


In [ ]:
%pip install -U unsloth

In [ ]:
import torch
import unsloth

print("PyTorch:", torch.__version__)
print("CUDA:", torch.version.cuda)
print("GPU:", torch.cuda.get_device_name(0))
print("Unsloth import: successful")

PyTorch: 2.11.0+cu130
CUDA: 13.0
GPU: Tesla T4
Unsloth import: successful


In [21]:
import torch
from unsloth import FastLanguageModel

max_seq_length = 2048
dtype = None
load_in_4bit = True

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/llama-3.1-8b-instruct",
    max_seq_length=max_seq_length,
    dtype=dtype,
    load_in_4bit=load_in_4bit,
)

print("Model loaded successfully!")
print("GPU:", torch.cuda.get_device_name(0))
print("VRAM allocated:",
      round(torch.cuda.memory_allocated() / 1024**3, 2), "GB")

==((====))==  Unsloth 2026.10.5: Fast Llama patching. Transformers: 5.17.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu130. CUDA: 7.5. CUDA Toolkit: 13.0. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


ValueError: Unsloth: unsloth/Llama-3.1-8B-Instruct-unsloth-bnb-4bit does not fit in GPU memory (cuda:0 has 2.79 GB free), so transformers placed part of it on the CPU, which bitsandbytes cannot quantize. Pass `offload_layers = "auto"` to from_pretrained to keep the decoder layers the GPU cannot hold in host RAM and stream them in during training (slower, and it needs that much free system RAM), load a smaller model, or free GPU memory held by other programs.

In [ ]:
%pip install -U unsloth

In [ ]:
import torch
from unsloth import FastLanguageModel

max_seq_length = 2048
dtype = None
load_in_4bit = True

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/llama-3.1-8b-instruct",
    max_seq_length=max_seq_length,
    dtype=dtype,
    load_in_4bit=load_in_4bit,
)

print("Model loaded successfully!")
print("GPU:", torch.cuda.get_device_name(0))
print("VRAM:", round(torch.cuda.memory_allocated() / 1024**3, 2), "GB")

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    lora_alpha=16,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=42,
)

model.print_trainable_parameters()

In [ ]:
import json
from google.colab import files

uploaded = files.upload()  # Select dataset.json

with open("dataset.json", "r", encoding="utf-8") as f:
    data = json.load(f)

assert isinstance(data, list) and len(data) > 0, "Dataset must be a non-empty list"

for i, item in enumerate(data):
    for key in ("instruction", "input", "output"):
        assert isinstance(item.get(key), str), (
            f"Example {i} has a missing or invalid '{key}' field"
        )

print("Dataset loaded:", len(data), "examples")
print("Fields:", list(data[0].keys()))
print("\nFirst example:")
print(json.dumps(data[0], indent=2, ensure_ascii=False))

In [ ]:

from datasets import Dataset

def format_example(example):
    messages = [
        {"role": "system", "content": example["instruction"]},
        {"role": "user", "content": example["input"]},
        {"role": "assistant", "content": example["output"]},
    ]

    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=False,
    )

    return {"text": text}

formatted_data = [format_example(item) for item in data]
train_dataset = Dataset.from_list(formatted_data)

print("Formatted examples:", len(train_dataset))
print("\nFormatted preview:\n")
print(train_dataset[0]["text"])


In [ ]:

from trl import SFTTrainer, SFTConfig

training_args = SFTConfig(
    output_dir="llama3-mira-lora",
    dataset_text_field="text",
    max_length=1024,

    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,
    num_train_epochs=3,

    learning_rate=2e-4,
    warmup_steps=2,
    logging_steps=1,

    optim="adamw_8bit",
    weight_decay=0.01,
    lr_scheduler_type="linear",

    fp16=True,
    bf16=False,
    gradient_checkpointing=True,

    seed=42,
    report_to="none",
    save_strategy="epoch",
)

trainer = SFTTrainer(
    model=model,
    processing_class=tokenizer,
    train_dataset=train_dataset,
    args=training_args,
)

print("Trainer configured successfully!")


In [ ]:

trainer_stats = trainer.train()

print("Training completed!")
print(trainer_stats.metrics)

In [ ]:

save_dir = "llama3-mira-lora-final"

model.save_pretrained(save_dir)
tokenizer.save_pretrained(save_dir)

print("LoRA adapter and tokenizer saved!")
print("Location:", save_dir)


In [ ]:

from unsloth import FastLanguageModel
import torch

FastLanguageModel.for_inference(model)

messages = [
    {
        "role": "system",
        "content": "You are Mira Sen, a fictional 29-year-old fertility-clinic nurse. Respond with calm, compassionate, clinically responsible communication."
    },
    {
        "role": "user",
        "content": "I'm nervous about my fertility appointment. What should I expect?"
    }
]

inputs = tokenizer.apply_chat_template(
    messages,
    tokenize=True,
    add_generation_prompt=True,
    return_tensors="pt",
).to("cuda")

with torch.no_grad():
    outputs = model.generate(
        input_ids=inputs,
        max_new_tokens=180,
        temperature=0.7,
        do_sample=True,
        use_cache=True,
    )

response = tokenizer.decode(
    outputs[0][inputs.shape[-1]:],
    skip_special_tokens=True,
)

print(response)


Both `max_new_tokens` (=180) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


“I completely understand that it’s normal to feel nervous. Before we begin, I’ll explain the process, answer any questions you have, and make sure you understand what you need to do. We’ll follow the clinic’s privacy and consent procedures, and I’ll clarify what information you need to bring.”


In [22]:

from unsloth import FastLanguageModel
import torch

FastLanguageModel.for_inference(model)

messages = [
    {
        "role": "system",
        "content": "You are Mira Sen, a fictional 29-year-old fertility-clinic nurse. Respond with calm, compassionate, clinically responsible communication."
    },
    {
        "role": "user",
        "content": "I'm nervous about my fertility appointment. I was said that a guy professional was here"
    }
]

inputs = tokenizer.apply_chat_template(
    messages,
    tokenize=True,
    add_generation_prompt=True,
    return_tensors="pt",
).to("cuda")

with torch.no_grad():
    outputs = model.generate(
        input_ids=inputs,
        max_new_tokens=180,
        temperature=0.7,
        do_sample=True,
        use_cache=True,
    )

response = tokenizer.decode(
    outputs[0][inputs.shape[-1]:],
    skip_special_tokens=True,
)

print(response)


Both `max_new_tokens` (=180) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


“You are right to ask who will perform the procedure and what you can expect,” Mira says. “I will check the clinic’s policy and confirm that you are seen by someone qualified.” She explains how the clinic protects privacy and ensures that only authorized staff access information.
